# 08장 보안 실습 — 지속성 위치와 승인 검토


## Goal

자동 시작 위치의 승인·실행 주체·수집 누락을 구분합니다.

[교안과 분석 질문](../../08-system-automation/08-3-persistence-review.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-08-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'persistence.psv': 'mechanism|path|owner|approval|observed_kst\nsystemd|/etc/systemd/system/report-helper.service|root|unknown|2026-09-10T09:05:00+09:00\ncron|/etc/cron.d/backup|root|CHG-100|2026-09-09T18:00:00+09:00\nshell-startup|/home/analyst/.bashrc|analyst|baseline|2026-09-01T10:00:00+09:00\nssh-key|/home/analyst/.ssh/authorized_keys|analyst|KEY-200|2026-09-01T10:00:00+09:00\n', 'service-review.txt': '# Synthetic review excerpt only. Not an installable unit file.\nunit=report-helper.service\nUser=collector\nExecStart=/opt/collector/bin/report\nFragmentPath=/etc/systemd/system/report-helper.service\nDropInPaths=not_collected\nchange_ticket=unknown\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: 승인 미확인 1개, 알려진 기록 3개, 설치한 지속성 0개

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. 자동 시작 위치의 승인 상태 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $4=="unknown" {print $1 "|" $2}' "$COURSE_DATA/persistence.psv" > "$COURSE_OUT/unapproved.psv"
test "$(wc -l < "$COURSE_OUT/unapproved.psv")" -eq 1
cat "$COURSE_OUT/unapproved.psv"


### 2. 서비스 실행 주체와 경로 읽기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -E '^(User|ExecStart|DropInPaths|change_ticket)=' "$COURSE_DATA/service-review.txt" > "$COURSE_OUT/service-context.txt"
grep -Fx 'User=collector' "$COURSE_OUT/service-context.txt"
grep -Fx 'DropInPaths=not_collected' "$COURSE_OUT/service-context.txt"


### 3. 정상 기준선도 함께 남기기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $4!="unknown" {print $1 "|" $4}' "$COURSE_DATA/persistence.psv" > "$COURSE_OUT/known.psv"
test "$(wc -l < "$COURSE_OUT/known.psv")" -eq 3
printf 'unknown_approval=1 known_records=3 installed_by_lab=0\n'


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
